# PMData Prompt Generation Guide

This notebook guides you through:
- Downloading/placing PMData in the expected layout
- Inspecting raw Fitbit and PMSys (wellness) samples
- Building the 14-day context windows
- Generating instruction/input/output triplets exactly as in `gen_dataset.py` for PMData

References:
- PMData: https://datasets.simula.no/pmdata/
- Kaggle analysis (Fitbit): https://www.kaggle.com/code/murderbydeath/how-i-analyzed-the-data-from-pmdata-fitbit
- Repo doc: `docs/PMData_README.md`


## Prerequisites & Data Layout

1) Request and download PMData from the official page (respect licensing/terms).
2) Unzip and place participant folders under `medalpaca/data/pmdata/` with the following structure:

```.text
medalpaca/data/pmdata/
  ├─ p1/
  │   ├─ fitbit/
  │   │   ├─ exercise.json
  │   │   ├─ resting_heart_rate.json
  │   │   └─ sleep.json
  │   └─ pmsys/
  │       └─ wellness.csv
  ├─ p2/
  │   └─ ...
  └─ ...
```

This notebook will not download PMData for you. Follow the PMData site instructions for access.


In [2]:
import os, json, csv, math
from datetime import datetime, timedelta
import pandas as pd
from glob import glob

BASE_PM_PATH = os.path.join('../medalpaca', 'data', 'pmdata')
assert os.path.isdir(BASE_PM_PATH), f'PMData base path not found: {BASE_PM_PATH}'
participants = sorted([d for d in os.listdir(BASE_PM_PATH) if os.path.isdir(os.path.join(BASE_PM_PATH, d))])
participants[:5]

['p01', 'p02', 'p03', 'p04', 'p05']

In [3]:
# Choose a participant
PID = participants[0] if participants else None
print('Using participant:', PID)
fb_dir = os.path.join(BASE_PM_PATH, PID, 'fitbit')
pm_dir = os.path.join(BASE_PM_PATH, PID, 'pmsys')
fb_dir, pm_dir

Using participant: p01


('../medalpaca/data/pmdata/p01/fitbit', '../medalpaca/data/pmdata/p01/pmsys')

In [4]:
# Load PMSys wellness CSV
wellness_csv = os.path.join(pm_dir, 'wellness.csv')
wdf = pd.read_csv(wellness_csv)
wdf.head()


,effective_time_frame,fatigue,mood,readiness,sleep_duration_h,sleep_quality,soreness,soreness_area,stress
0,2019-11-01T08:31:40.751Z,2,3,5,6,3,2,[12921003],3
1,2019-11-02T10:00:01.229Z,2,3,6,6,3,2,[12921003],3
2,2019-11-03T14:28:03.263Z,3,3,8,6,3,3,[],3
3,2019-11-04T07:05:28.429Z,3,3,8,6,3,3,[],3
4,2019-11-05T06:13:35.998Z,3,3,8,5,3,3,[],3


In [5]:
# Parse fields as expected by gen_dataset.py
def parse_effective_time_frame(s):
    # example format like: 'YYYY-MM-DDTHH:MM:SS.sssZ' -> used as 'YYYY-MM-DD_HH:MM:SS'
    d = s[:10] + '_' + s[11:].split('.')[0].rstrip('Z')
    return datetime.strptime(d, '%Y-%m-%d_%H:%M:%S')

wdf['_ts'] = wdf['effective_time_frame'].apply(parse_effective_time_frame)
wdf[['effective_time_frame','_ts','fatigue','mood','readiness','sleep_duration_h','sleep_quality','stress']].head()


,effective_time_frame,_ts,fatigue,mood,readiness,sleep_duration_h,sleep_quality,stress
0,2019-11-01T08:31:40.751Z,2019-11-01 08:31:40,2,3,5,6,3,3
1,2019-11-02T10:00:01.229Z,2019-11-02 10:00:01,2,3,6,6,3,3
2,2019-11-03T14:28:03.263Z,2019-11-03 14:28:03,3,3,8,6,3,3
3,2019-11-04T07:05:28.429Z,2019-11-04 07:05:28,3,3,8,6,3,3
4,2019-11-05T06:13:35.998Z,2019-11-05 06:13:35,3,3,8,5,3,3


In [6]:
# Pick a wellness row (target date)
row = wdf.iloc[0]
target_dt = row['_ts']
print('Target date:', target_dt)
mood = row['mood']
labels = {
    'readiness': row['readiness'],
    'stress': row['stress'],
    'sleep_quality': row['sleep_quality'],
    'fatigue': row['fatigue'],
}
labels


Target date: 2019-11-01 08:31:40


{'readiness': np.int64(5),
 'stress': np.int64(3),
 'sleep_quality': np.int64(3),
 'fatigue': np.int64(2)}

In [7]:
# Load Fitbit JSONs
with open(os.path.join(fb_dir, 'exercise.json')) as f: exercise = json.load(f)
with open(os.path.join(fb_dir, 'resting_heart_rate.json')) as f: rhr = json.load(f)
with open(os.path.join(fb_dir, 'sleep.json')) as f: sleep = json.load(f)
len(exercise), len(rhr), len(sleep)


(190, 152, 155)

In [22]:
# rhr[0]

In [23]:
# exe_0 = exercise[0]
# ed = datetime.strptime(exe_0['startTime'][:10] + '_' + exe_0['startTime'][11:], '%Y-%m-%d_%H:%M:%S')
# print(ed)
# print(target_dt)
# print(ed-target_dt)

In [24]:
# Helper: time window filter
def in_window(ts, ref, days=14):
    return (ref > ts) and ((ref - ts) < timedelta(days=days))

# Collect 14-day histories as in gen_dataset.py
exercise_hist = []  # [date, activity, duration(min), calories, steps]
for c, e in enumerate(exercise):
    ed = datetime.strptime(e['startTime'][:10] + '_' + e['startTime'][11:], '%Y-%m-%d_%H:%M:%S')
    if in_window(ed, target_dt, 14):
        try:
            activity = e['activityName']
            burn_calories = float(e['calories'])
            steps = float(e['steps'])
            duration_min = float(e['duration']) / 1000.0 / 60.0
            exercise_hist.append([ed, activity, duration_min, burn_calories, steps])
        except Exception:
            pass

sleep_hist = []   # [date, sleep_minutes]
for s in sleep:
    sd = datetime.strptime(s['startTime'][:10] + '_' + s['startTime'][11:], '%Y-%m-%d_%H:%M:%S')
    if in_window(sd, target_dt, 14):
        sleep_min = float(s['duration']) / 1000.0 / 60.0
        sleep_hist.append([sd, sleep_min])

hr_hist = []      # [date, rhr]
for h in rhr:
    hd = datetime.strptime(h['dateTime'][:10] + '_' + h['dateTime'][11:], '%Y-%m-%d_%H:%M:%S')
    if in_window(hd, target_dt, 14):
        rhr_val = float(h['value']['value'])
        hr_hist.append([hd, rhr_val])

len(exercise_hist), len(sleep_hist), len(hr_hist)


(0, 0, 1)

In [25]:
# Show a few raw samples for sanity
print('Exercise sample:', exercise_hist[:2])
print('Sleep sample   :', sleep_hist[:2])
print('RHR sample     :', hr_hist[:2])


Exercise sample: []
Sleep sample   : []
RHR sample     : [[datetime.datetime(2019, 11, 1, 0, 0), 53.74107360839844]]


In [26]:
# Build prompt components exactly as in gen_dataset.py
SUBTASK = 'sleep_quality'  # choose from: 'sleep_quality', 'stress', 'readiness', 'fatigue'
ranges = {
    'readiness': (0, 10),
    'stress': (1, 5),
    'sleep_quality': (1, 5),
    'fatigue': (1, 5),
}
r1, r2 = ranges[SUBTASK]
I = f'You are a personalized healthcare agent trained to predict {SUBTASK} which ranges from {r1} to {r2} based on physiological data and user information.'
steps_list = [x[-1] for x in exercise_hist]
cals_list = [x[-2] for x in exercise_hist]
rhr_list  = [x[-1] for x in hr_hist]
sleep_list= [x[-1] for x in sleep_hist]
Q = (
    f'The recent 14-days sensor readings show: [Steps]: {steps_list} steps, ' +
    f'[Burned Calorories]: {cals_list} calories, ' +
    f'[Resting Heart Rate]: {rhr_list} beats/min, ' +
    f'[SleepMinutes]: {sleep_list} minutes, ' +
    f'[Mood]: {mood} out of 5; What would be the predicted {SUBTASK}?'
)
A = f'The predicted {SUBTASK} level is {labels[SUBTASK]}.'
print('Instruction:', I)
print('Input:', Q)
print('Output:', A)


Instruction: You are a personalized healthcare agent trained to predict sleep_quality which ranges from 1 to 5 based on physiological data and user information.
Input: The recent 14-days sensor readings show: [Steps]: [] steps, [Burned Calorories]: [] calories, [Resting Heart Rate]: [53.74107360839844] beats/min, [SleepMinutes]: [] minutes, [Mood]: 3 out of 5; What would be the predicted sleep_quality?
Output: The predicted sleep_quality level is 3.


### Optional: Aggregated-stats variant (for readability)
Not used in the paper’s released code, but helpful for human inspection.

In [ ]:
import statistics as stats
def safe_mean(xs):
    return round(stats.mean(xs), 2) if xs else 'N/A'
Q_stats = (
    f"The recent 14-days sensor readings show averages: "
    f"Steps: {safe_mean(steps_list)}, Calories: {safe_mean(cals_list)}, "
    f"Resting HR: {safe_mean(rhr_list)}, Sleep Minutes: {safe_mean(sleep_list)}; "
    f"Mood: {mood} out of 5. What would be the predicted {SUBTASK}?"
)
print(Q_stats)


In [ ]:
# Package one example as JSON if needed
example = {
    'instruction': I,
    'input': Q,
    'output': A,
}
example
